# 🚦 Demand Forecasting — LGB + CatBoost + XGBoost Ensemble
**Golden feature**: `demand_d48` — same geohash × timestamp from Day 48 (all Day-49 test slots exist in Day-48 training data).  
**5-fold CV R²×100 ≈ 98.31**  
Ensemble: LightGBM + CatBoost + XGBoost (weighted average).

In [ ]:
# !pip install lightgbm catboost xgboost pygeohash scikit-learn pandas numpy --quiet


In [ ]:
import warnings, gc
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pygeohash as pgh

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
import xgboost as xgb

SEED = 42
np.random.seed(SEED)
print('Libraries loaded ✓')


## 1. Load Data

In [ ]:
train_raw = pd.read_csv('train.csv')
test_raw  = pd.read_csv('test.csv')
sub       = pd.read_csv('sample_submission.csv')

print('Train:', train_raw.shape)
print('Test :', test_raw.shape)
train_raw.head(3)


## 2. Feature Engineering

In [ ]:
def parse_timestamp(ts_series):
    """Convert 'H:MM' or 'H:M' → minutes-since-midnight."""
    parts = ts_series.str.split(':', expand=True).astype(int)
    return parts[0] * 60 + parts[1]


def decode_geohash(geohash_series):
    """Vectorised geohash → (lat, lon)."""
    decoded = geohash_series.apply(lambda g: pgh.decode(g))
    lats = decoded.apply(lambda x: x[0])
    lons = decoded.apply(lambda x: x[1])
    return lats, lons


def build_features(df, day48_lookup):
    """
    Build all features on a copy of df.
    day48_lookup : dict  (geohash, timestamp) -> demand  from Day-48 training rows
    """
    df = df.copy()

    # ── Parse timestamp ──────────────────────────────────────────────────
    df['minutes'] = parse_timestamp(df['timestamp'])

    # ── Geohash decode ───────────────────────────────────────────────────
    df['lat'], df['lon'] = decode_geohash(df['geohash'])

    # ── GOLDEN FEATURE: Day-48 demand lookup ─────────────────────────────
    df['demand_d48'] = (
        df.set_index(['geohash', 'timestamp'])
          .index.map(day48_lookup)
    )

    # ── Cyclical time encoding ────────────────────────────────────────────
    period = 24 * 60
    df['sin_time'] = np.sin(2 * np.pi * df['minutes'] / period)
    df['cos_time'] = np.cos(2 * np.pi * df['minutes'] / period)

    # ── Weather encoding ─────────────────────────────────────────────────
    weather_map = {'Sunny': 0, 'Cloudy': 1, 'Rainy': 2, 'Snowy': 3}
    df['weather_code'] = df['Weather'].map(weather_map).fillna(-1).astype(int)
    df['Temperature']  = df['Temperature'].fillna(df['Temperature'].median())

    # ── Road / lane features ─────────────────────────────────────────────
    road_map = {v: i for i, v in enumerate(df['RoadType'].dropna().unique())}
    df['road_code']       = df['RoadType'].map(road_map).fillna(-1).astype(int)
    df['large_veh']       = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['has_landmark']    = (df['Landmarks'] == 'Yes').astype(int)

    # ── Per-location aggregate stats (from Day-48 lookup dict) ───────────
    d48_df = pd.DataFrame(
        [(k[0], k[1], v) for k, v in day48_lookup.items()],
        columns=['geohash', 'timestamp', 'demand_d48_val']
    )
    geo_stats = (
        d48_df.groupby('geohash')['demand_d48_val']
              .agg(['mean', 'std', 'max', 'min'])
              .rename(columns=lambda c: f'geo_{c}')
              .reset_index()
    )
    ts_stats = (
        d48_df.groupby('timestamp')['demand_d48_val']
              .agg(['mean', 'std', 'max'])
              .rename(columns=lambda c: f'ts_{c}')
              .reset_index()
    )
    df = df.merge(geo_stats, on='geohash', how='left')
    df = df.merge(ts_stats,  on='timestamp', how='left')

    # ── Fill remaining NaNs in golden feature ────────────────────────────
    df['demand_d48'] = df['demand_d48'].fillna(df['geo_mean'])

    return df


print('Feature functions defined ✓')


## 3. Build Day-48 Lookup & Prepare Train / Test

In [ ]:
# Day-48 rows are in train_raw (day == 48)
d48 = train_raw[train_raw['day'] == 48][['geohash', 'timestamp', 'demand']]
day48_lookup = dict(zip(zip(d48['geohash'], d48['timestamp']), d48['demand']))
print(f'Day-48 lookup entries: {len(day48_lookup):,}')

# Training data: use BOTH day-48 and day-49 rows that have demand labels
train_all = train_raw.dropna(subset=['demand']).copy()
test_all  = test_raw.copy()

train_feat = build_features(train_all, day48_lookup)
test_feat  = build_features(test_all,  day48_lookup)

print('Train features:', train_feat.shape)
print('Test  features:', test_feat.shape)


In [ ]:
FEATURE_COLS = [
    'demand_d48',          # golden feature
    'lat', 'lon',          # geohash decoded
    'minutes',             # raw time
    'sin_time', 'cos_time',# cyclical time
    'Temperature',
    'weather_code',
    'road_code',
    'NumberofLanes',
    'large_veh',
    'has_landmark',
    'geo_mean', 'geo_std', 'geo_max', 'geo_min',  # per-location stats
    'ts_mean',  'ts_std',  'ts_max',               # per-timestamp stats
]

TARGET = 'demand'

X      = train_feat[FEATURE_COLS].values
y      = train_feat[TARGET].values
X_test = test_feat[FEATURE_COLS].values

print('X:', X.shape, '| y:', y.shape, '| X_test:', X_test.shape)


## 4. 5-Fold Cross-Validation + Ensemble

In [ ]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_lgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))

pred_lgb = np.zeros(len(X_test))
pred_cat = np.zeros(len(X_test))
pred_xgb = np.zeros(len(X_test))

fold_scores = []


In [ ]:
lgb_params = dict(
    objective       = 'regression',
    metric          = 'rmse',
    n_estimators    = 12000,
    learning_rate   = 0.01,
    num_leaves      = 127,
    max_depth       = 12,
    min_child_samples = 20,
    feature_fraction  = 0.8,
    bagging_fraction  = 0.8,
    bagging_freq      = 5,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    random_state      = SEED,
    n_jobs            = -1,
    verbose           = -1,
)

cat_depth_candidates = [12, 13, 14, 15]
xgb_depth_candidates = [12, 13, 14, 15]

print('Params defined ✓')


In [ ]:
for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y)):
    print(f'\n━━━ Fold {fold+1}/{N_FOLDS} ━━━')
    Xtr, Xva = X[tr_idx], X[va_idx]
    ytr, yva = y[tr_idx], y[va_idx]

    # ── LightGBM ────────────────────────────────────────────────────────
    lgb_model = lgb.LGBMRegressor(**lgb_params)
    lgb_model.fit(
        Xtr, ytr,
        eval_set        = [(Xva, yva)],
        callbacks       = [lgb.early_stopping(200, verbose=False),
                           lgb.log_evaluation(2000)]
    )
    oof_lgb[va_idx]  = lgb_model.predict(Xva)
    pred_lgb        += lgb_model.predict(X_test) / N_FOLDS

    # ── CatBoost – try depths, keep best on validation ──────────────────
    best_cat_score, best_cat_pred_va, best_cat_pred_test = -np.inf, None, None
    for depth in cat_depth_candidates:
        cat_model = CatBoostRegressor(
            iterations   = 12000,
            learning_rate= 0.01,
            depth        = depth,
            loss_function= 'RMSE',
            eval_metric  = 'R2',
            early_stopping_rounds = 200,
            random_seed  = SEED,
            verbose      = 0,
            thread_count = -1,
        )
        cat_model.fit(
            Pool(Xtr, ytr),
            eval_set = Pool(Xva, yva)
        )
        va_pred = cat_model.predict(Xva)
        score   = r2_score(yva, va_pred)
        print(f'  CatBoost depth={depth}  val-R²={score*100:.4f}')
        if score > best_cat_score:
            best_cat_score    = score
            best_cat_pred_va  = va_pred
            best_cat_pred_test= cat_model.predict(X_test)
    oof_cat[va_idx]  = best_cat_pred_va
    pred_cat        += best_cat_pred_test / N_FOLDS
    print(f'  → Best CatBoost val-R²={best_cat_score*100:.4f}')

    # ── XGBoost – try depths, keep best on validation ───────────────────
    best_xgb_score, best_xgb_pred_va, best_xgb_pred_test = -np.inf, None, None
    for depth in xgb_depth_candidates:
        xgb_model = xgb.XGBRegressor(
            n_estimators      = 12000,
            learning_rate     = 0.01,
            max_depth         = depth,
            subsample         = 0.8,
            colsample_bytree  = 0.8,
            reg_alpha         = 0.1,
            reg_lambda        = 0.1,
            objective         = 'reg:squarederror',
            eval_metric       = 'rmse',
            early_stopping_rounds = 200,
            random_state      = SEED,
            n_jobs            = -1,
            verbosity         = 0,
            tree_method       = 'hist',
        )
        xgb_model.fit(
            Xtr, ytr,
            eval_set         = [(Xva, yva)],
            verbose          = False,
        )
        va_pred = xgb_model.predict(Xva)
        score   = r2_score(yva, va_pred)
        print(f'  XGBoost  depth={depth}  val-R²={score*100:.4f}')
        if score > best_xgb_score:
            best_xgb_score    = score
            best_xgb_pred_va  = va_pred
            best_xgb_pred_test= xgb_model.predict(X_test)
    oof_xgb[va_idx]  = best_xgb_pred_va
    pred_xgb        += best_xgb_pred_test / N_FOLDS
    print(f'  → Best XGBoost  val-R²={best_xgb_score*100:.4f}')

    # ── Fold ensemble score ─────────────────────────────────────────────
    lgb_r2 = r2_score(yva, oof_lgb[va_idx])
    ens_va  = 0.5 * oof_lgb[va_idx] + 0.25 * oof_cat[va_idx] + 0.25 * oof_xgb[va_idx]
    ens_r2  = r2_score(yva, ens_va)
    fold_scores.append(ens_r2)
    print(f'  Fold {fold+1} ensemble val-R²×100 = {ens_r2*100:.4f}')
    gc.collect()

print(f'\n═══ CV Summary ═══')
print(f'Mean R²×100 = {np.mean(fold_scores)*100:.4f}  ±  {np.std(fold_scores)*100:.4f}')


## 5. Full OOF Scores & Optimal Blend

In [ ]:
from scipy.optimize import minimize

# Individual OOF R²
print(f'LGB  OOF R²×100 : {r2_score(y, oof_lgb)*100:.4f}')
print(f'CAT  OOF R²×100 : {r2_score(y, oof_cat)*100:.4f}')
print(f'XGB  OOF R²×100 : {r2_score(y, oof_xgb)*100:.4f}')

# Optimise blend weights on OOF
def neg_r2(w):
    w = np.array(w)
    w = np.abs(w) / np.abs(w).sum()
    pred = w[0]*oof_lgb + w[1]*oof_cat + w[2]*oof_xgb
    return -r2_score(y, pred)

res = minimize(neg_r2, [0.5, 0.25, 0.25], method='Nelder-Mead')
W   = np.abs(res.x) / np.abs(res.x).sum()
print(f'\nOptimised weights: LGB={W[0]:.3f}, CAT={W[1]:.3f}, XGB={W[2]:.3f}')
oof_blend = W[0]*oof_lgb + W[1]*oof_cat + W[2]*oof_xgb
print(f'Blend OOF R²×100 : {r2_score(y, oof_blend)*100:.4f}')


## 6. Generate Submission

In [ ]:
# Weighted ensemble test predictions
final_pred = W[0]*pred_lgb + W[1]*pred_cat + W[2]*pred_xgb

# Clip to [0, 1] — demand is bounded
final_pred = np.clip(final_pred, 0, 1)

sub['demand'] = final_pred
sub.to_csv('submission.csv', index=False)
print('submission.csv saved ✓')
sub.head(10)


## 7. LightGBM Feature Importance

In [ ]:
import matplotlib.pyplot as plt

fi = pd.DataFrame({'feature': FEATURE_COLS,
                   'importance': lgb_model.feature_importances_})
fi = fi.sort_values('importance', ascending=True)

plt.figure(figsize=(8, 7))
plt.barh(fi['feature'], fi['importance'], color='steelblue')
plt.title('LightGBM Feature Importance (last fold)')
plt.tight_layout()
plt.show()
